# Azure AI Search Pipeline Setup

Build an Azure AI Search indexing pipeline using the Python SDK (`azure-search-documents`).

This performs the same operations as the REST API (.rest files 01–04), but via the Python SDK.

| Step | Description |
|------|-------------|
| 1 | **Data Source** — Azure Blob Storage (Managed Identity) |
| 2 | **Index** — Vector Search + Semantic Search + Scalar Quantization |
| 3 | **Skillset** — ContentUnderstandingSkill + ChatCompletionSkill + Embeddings + Index Projections |
| 4 | **Indexer** — Run the pipeline |

### References
- [ContentUnderstandingSkill (Python SDK)](https://learn.microsoft.com/en-us/python/api/azure-search-documents/azure.search.documents.indexes.models.contentunderstandingskill?view=azure-python-preview)
- [ChatCompletionSkill (Python SDK)](https://learn.microsoft.com/en-us/python/api/azure-search-documents/azure.search.documents.indexes.models.chatcompletionskill?view=azure-python-preview)


In [ ]:
# The preview version of azure-search-documents is required
# (ContentUnderstandingSkill, ChatCompletionSkill are preview APIs)
%pip install azure-search-documents --pre azure-identity python-dotenv --quiet

## Configuration

Set up connection details and resource names.  
API keys are loaded from environment variables or a `.env` file.

In [ ]:
import os
from datetime import timedelta
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    # Index
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    # Vector search
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    ScalarQuantizationCompression,
    ScalarQuantizationParameters,
    RescoringOptions,
    # Semantic search
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
    # Data source
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection,
    # Skillset skills
    ContentUnderstandingSkill,
    ContentUnderstandingSkillChunkingProperties,
    AzureOpenAIEmbeddingSkill,
    ChatCompletionSkill,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    # Skillset structure
    SearchIndexerSkillset,
    AIServicesAccountKey,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    IndexProjectionMode,
    # Indexer
    SearchIndexer,
    FieldMapping,
)
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

load_dotenv(override=True)

# ── Azure AI Search ──────────────────────────────────────
search_endpoint = os.environ.get("AZURE_SEARCH_ENDPOINT", "https://demo-cogsearch01.search.windows.net")
search_api_key  = os.environ["AZURE_SEARCH_API_KEY"]
credential = AzureKeyCredential(search_api_key)

# ── Azure OpenAI ─────────────────────────────────────────
aoai_endpoint   = os.environ.get("AZURE_OPENAI_ENDPOINT", "https://aifoundry05.openai.azure.com")
aoai_api_key    = os.environ["AZURE_OPENAI_API_KEY"]
embedding_deployment = "text-embedding-3-large"
embedding_model     = "text-embedding-3-large"
embedding_dimensions = 3072
chat_deployment     = os.environ.get("CHAT_DEPLOYMENT", "gpt-5")

# ── AI Services (for ContentUnderstandingSkill) ─────────
ai_services_key = os.environ["AI_SERVICES_KEY"]
ai_services_url = os.environ["AI_SERVICES_SUBDOMAIN_URL"]

# ── Storage (Managed Identity via ResourceId) ────────────
storage_connection_string = os.environ["STORAGE_CONNECTION_STRING"]
container_name = "disneyland-map"

# ── Resource Names ────────────────────────────────────────
datasource_name = "ks-disneymap-datasource"
index_name      = "ks-disneymap-index"
skillset_name   = "ks-disneymap-skillset"
indexer_name    = "ks-disneymap-indexer"

print("Configuration loaded.")
print(f"  Search endpoint : {search_endpoint}")
print(f"  AOAI endpoint   : {aoai_endpoint}")
print(f"  Index name      : {index_name}")

## Step 1 — Create Data Source

Define the connection to Azure Blob Storage.  
The connection string uses Managed Identity (ResourceId).

In [ ]:
indexer_client = SearchIndexerClient(endpoint=search_endpoint, credential=credential)

data_source = SearchIndexerDataSourceConnection(
    name=datasource_name,
    description="Data source for knowledge source 'ks-disneymap-user-guide'",
    type="azureblob",
    connection_string=storage_connection_string,
    container=SearchIndexerDataContainer(name=container_name),
)

result = indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source '{result.name}' created or updated.")

## Step 2 — Create Index

Create the search index.

| Field | Purpose |
|-------|---------|
| `uid` | Document key (keyword analyzer) |
| `snippet_parent_id` | Parent document ID for text chunks |
| `image_snippet_parent_id` | Parent document ID for image chunks |
| `snippet` | Text / image description (ja.microsoft analyzer) |
| `snippet_vector` | Vector embedding (3072 dimensions) |
| `blob_url` | Source file URL |

Vector Search: HNSW + Scalar Quantization (int8) + Azure OpenAI Vectorizer  
Semantic Search: snippet field set as prioritized content

In [ ]:
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)

# ── Field Definitions ─────────────────────────────────────
fields = [
    SearchField(
        name="uid",
        type=SearchFieldDataType.String,
        key=True,
        searchable=True,
        filterable=False,
        retrievable=True,
        stored=True,
        sortable=True,
        facetable=False,
        analyzer_name="keyword",
    ),
    SearchField(
        name="snippet_parent_id",
        type=SearchFieldDataType.String,
        searchable=False,
        filterable=True,
        retrievable=True,
        stored=True,
        sortable=False,
        facetable=False,
    ),
    SearchField(
        name="blob_url",
        type=SearchFieldDataType.String,
        searchable=False,
        filterable=True,
        retrievable=True,
        stored=True,
        sortable=False,
        facetable=False,
    ),
    SearchField(
        name="snippet",
        type=SearchFieldDataType.String,
        searchable=True,
        filterable=False,
        retrievable=True,
        stored=True,
        sortable=False,
        facetable=False,
        analyzer_name="ja.microsoft",
    ),
    SearchField(
        name="image_snippet_parent_id",
        type=SearchFieldDataType.String,
        searchable=False,
        filterable=True,
        retrievable=True,
        stored=True,
        sortable=False,
        facetable=False,
    ),
    SearchField(
        name="snippet_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        filterable=False,
        retrievable=True,
        stored=True,
        sortable=False,
        facetable=False,
        vector_search_dimensions=embedding_dimensions,
        vector_search_profile_name=f"{index_name}-vector-search-profile",
    ),
]

# ── Vector Search Configuration ───────────────────────────
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name=f"{index_name}-vector-search-algorithm",
            parameters=HnswParameters(
                metric="cosine",
                m=4,
                ef_construction=400,
                ef_search=500,
            ),
        ),
    ],
    profiles=[
        VectorSearchProfile(
            name=f"{index_name}-vector-search-profile",
            algorithm_configuration_name=f"{index_name}-vector-search-algorithm",
            vectorizer_name=f"{index_name}-vectorizer",
            compression_name=f"{index_name}-vector-search-scalar-quantization",
        ),
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            vectorizer_name=f"{index_name}-vectorizer",
            parameters=AzureOpenAIVectorizerParameters(
                resource_url=aoai_endpoint,
                deployment_name=embedding_deployment,
                model_name=embedding_model,
                api_key=aoai_api_key,
            ),
        ),
    ],
    compressions=[
        ScalarQuantizationCompression(
            compression_name=f"{index_name}-vector-search-scalar-quantization",
            parameters=ScalarQuantizationParameters(quantized_data_type="int8"),
            rescoring_options=RescoringOptions(
                enable_rescoring=True,
                default_oversampling=4.0,
                rescore_storage_method="preserveOriginals",
            ),
        ),
    ],
)

# ── Semantic Search Configuration ─────────────────────────
semantic_config = SemanticConfiguration(
    name=f"{index_name}-semantic-configuration",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="snippet")],
    ),
)
semantic_search = SemanticSearch(
    default_configuration_name=f"{index_name}-semantic-configuration",
    configurations=[semantic_config],
)

# ── Create Index ──────────────────────────────────────────
index = SearchIndex(
    name=index_name,
    description="Search index for knowledge source 'ks-disneymap-user-guide'",
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' created or updated.")

## Step 3 — Create Skillset

Create the skillset, composed of the following 4 skills:

| # | Skill | Description |
|---|-------|-------------|
| 1 | **ContentUnderstandingSkill** | Extract text chunks and images from documents |
| 2 | **AzureOpenAIEmbeddingSkill** | Generate vector embeddings for text chunks |
| 3 | **ChatCompletionSkill** | Verbalize images — generate descriptions in Japanese |
| 4 | **AzureOpenAIEmbeddingSkill** | Generate vector embeddings for verbalized image descriptions |

Index Projections project both text chunks and image descriptions into the same index.

In [ ]:
# ── System Message (for ChatCompletionSkill) ─────────────
system_message = (
    "You are tasked with generating concise and accurate descriptions of images, "
    "figures, diagrams, or charts in documents. When a document contains a map, "
    "include information about the nearest positional relationships between numbered "
    "items within the map area (i.e., only those located in the green area). For "
    "example, note that numbers 60, 63, and 53 are located near number 52. For the "
    "numbered labels on the map, describe their relative positions in a systematic "
    "order, scanning from left to right and then from top to bottom. All descriptions "
    "must be written in Japanese."
)

# ── Skill 1: ContentUnderstandingSkill ────────────────────
content_understanding_skill = ContentUnderstandingSkill(
    name="contentUnderstandingSkill",
    context="/document",
    extraction_options=["images", "locationMetadata"],
    inputs=[
        InputFieldMappingEntry(name="file_data", source="/document/file_data"),
    ],
    outputs=[
        OutputFieldMappingEntry(name="text_sections", target_name="text_sections"),
        OutputFieldMappingEntry(name="normalized_images", target_name="normalized_images"),
    ],
    chunking_properties=ContentUnderstandingSkillChunkingProperties(
        unit="characters",
        maximum_length=2000,
        overlap_length=200,
    ),
)

# ── Skill 2: Embedding for Text Chunks ───────────────────
text_embedding_skill = AzureOpenAIEmbeddingSkill(
    name="AzureOpenAIEmbeddingSkill",
    description="Generate embeddings",
    context="/document/text_sections/*",
    resource_url=aoai_endpoint,
    api_key=aoai_api_key,
    deployment_name=embedding_deployment,
    model_name=embedding_model,
    dimensions=embedding_dimensions,
    inputs=[
        InputFieldMappingEntry(name="text", source="/document/text_sections/*/content"),
    ],
    outputs=[
        OutputFieldMappingEntry(name="embedding", target_name="text_vector"),
    ],
)

# ── Skill 3: ChatCompletionSkill (Image → Japanese Desc.) ─
chat_uri = (
    f"{aoai_endpoint}/openai/deployments/{chat_deployment}"
    "/chat/completions?api-version=2024-10-21"
)

chat_completion_skill = ChatCompletionSkill(
    name="GenAISkill",
    description="Generate chat responses for image verbalization",
    context="/document/normalized_images/*",
    uri=chat_uri,
    http_method="POST",
    timeout=timedelta(minutes=3, seconds=50),  # Max: PT3M50S
    batch_size=1,
    api_key=aoai_api_key,
    inputs=[
        InputFieldMappingEntry(name="systemMessage", source=f"='{system_message}'"),
        InputFieldMappingEntry(name="userMessage", source="='Please describe this image.'"),
        InputFieldMappingEntry(name="image", source="/document/normalized_images/*/data"),
    ],
    outputs=[
        OutputFieldMappingEntry(name="response", target_name="verbalizedImage"),
    ],
)

# ── Skill 4: Embedding for Verbalized Image Descriptions ─
image_embedding_skill = AzureOpenAIEmbeddingSkill(
    name="VerbalizedImageAzureOpenAIEmbeddingSkill",
    description="Generate embeddings",
    context="/document/normalized_images/*",
    resource_url=aoai_endpoint,
    api_key=aoai_api_key,
    deployment_name=embedding_deployment,
    model_name=embedding_model,
    dimensions=embedding_dimensions,
    inputs=[
        InputFieldMappingEntry(
            name="text",
            source="/document/normalized_images/*/verbalizedImage",
        ),
    ],
    outputs=[
        OutputFieldMappingEntry(name="embedding", target_name="verbalizedImage_vector"),
    ],
)

# ── Index Projections ─────────────────────────────────────
# Project both text chunks and image descriptions into the same index
index_projections = SearchIndexerIndexProjection(
    selectors=[
        # For text chunks
        SearchIndexerIndexProjectionSelector(
            target_index_name=index_name,
            parent_key_field_name="snippet_parent_id",
            source_context="/document/text_sections/*",
            mappings=[
                InputFieldMappingEntry(
                    name="snippet_vector",
                    source="/document/text_sections/*/text_vector",
                ),
                InputFieldMappingEntry(
                    name="snippet",
                    source="/document/text_sections/*/content",
                ),
                InputFieldMappingEntry(
                    name="blob_url",
                    source="/document/blob_url",
                ),
            ],
        ),
        # For image descriptions
        SearchIndexerIndexProjectionSelector(
            target_index_name=index_name,
            parent_key_field_name="image_snippet_parent_id",
            source_context="/document/normalized_images/*",
            mappings=[
                InputFieldMappingEntry(
                    name="snippet_vector",
                    source="/document/normalized_images/*/verbalizedImage_vector",
                ),
                InputFieldMappingEntry(
                    name="snippet",
                    source="/document/normalized_images/*/verbalizedImage",
                ),
                InputFieldMappingEntry(
                    name="blob_url",
                    source="/document/blob_url",
                ),
            ],
        ),
    ],
    parameters=SearchIndexerIndexProjectionsParameters(
        projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS,
    ),
)

# ── AI Services (AIServicesAccountKey + subdomainUrl) ─────
cognitive_services_account = AIServicesAccountKey(
    key=ai_services_key,
    subdomain_url=ai_services_url,
    description="AI Services for knowledge source",
)

# ── Create Skillset ───────────────────────────────────────
skillset = SearchIndexerSkillset(
    name=skillset_name,
    description="Skillset for knowledge source 'ks-disneymap-user-guide'",
    skills=[
        content_understanding_skill,
        text_embedding_skill,
        chat_completion_skill,
        image_embedding_skill,
    ],
    cognitive_services_account=cognitive_services_account,
    index_projection=index_projections,
)

result = indexer_client.create_or_update_skillset(skillset)
print(f"Skillset '{result.name}' created or updated.")

## Step 4 — Create and Run the Indexer

Create and run the indexer.  
`allowSkillsetToReadFileData: true` enables the skillset to read file data (e.g., images).

In [ ]:
indexer = SearchIndexer(
    name=indexer_name,
    description="Indexer for knowledge source 'ks-disneymap-user-guide'",
    data_source_name=datasource_name,
    skillset_name=skillset_name,
    target_index_name=index_name,
    parameters={
        "maxFailedItems": -1,
        "maxFailedItemsPerBatch": -1,
        "configuration": {
            "dataToExtract": "contentAndMetadata",
            "parsingMode": "default",
            "allowSkillsetToReadFileData": True,
        },
    },
    field_mappings=[
        FieldMapping(
            source_field_name="metadata_storage_path",
            target_field_name="blob_url",
        ),
    ],
)

result = indexer_client.create_or_update_indexer(indexer)
print(f"Indexer '{result.name}' created or updated.")
print("Indexer is running. Please wait a few minutes before querying.")

## Check Indexer Status

In [ ]:
# Check indexer execution status
status = indexer_client.get_indexer_status(indexer_name)
print(f"Status          : {status.status}")
print(f"Last result     : {status.last_result.status if status.last_result else 'N/A'}")
if status.last_result:
    print(f"  Items succeeded : {status.last_result.item_count}")
    print(f"  Items failed    : {status.last_result.failed_item_count}")
    print(f"  Start time      : {status.last_result.start_time}")
    print(f"  End time        : {status.last_result.end_time}")
    if status.last_result.errors:
        for err in status.last_result.errors:
            print(f"  Error: {err.error_message}")

## Search Test

After the indexer finishes running, test with vector search + semantic search.

In [ ]:
query = "What facility is located across from number 7 in World Bazaar?"

search_client = SearchClient(
    endpoint=search_endpoint,
    credential=credential,
    index_name=index_name,
)
vector_query = VectorizableTextQuery(
    text=query,
    k_nearest_neighbors=50,
    fields="snippet_vector",
)

results = search_client.search(
    search_text=query,
    vector_queries=[vector_query],
    select=["snippet", "blob_url"],
    top=3,
)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} (score: {result['@search.score']:.4f}) ---")
    print(f"blob_url: {result.get('blob_url', 'N/A')}")
    snippet = result.get("snippet", "")
    print(f"snippet : {snippet[:300]}..." if len(snippet) > 300 else f"snippet : {snippet}")